# TP — Régression linéaire multiple : pourquoi la mortalité du COVID a-t-elle tant varié d'un pays à l'autre ?

Dans le TP précédent, vous avez prédit une réponse $y$ à partir d'**une** variable
explicative. Ici, on en aura **huit**, et la question devient : *est-ce qu'ajouter des
variables améliore vraiment les prédictions ?*

Les données viennent d'**Our World in Data** : une ligne par pays, avec le bilan de
l'épidémie et une série de caractéristiques du pays — âge médian, PIB par habitant,
espérance de vie, lits d'hôpitaux, etc.

On cherche à prédire le **nombre total de décès par million d'habitants**.

**Le fil du TP** — volontairement le même que le précédent :

1. on charge et on nettoie les données ;
2. on sépare train et test ;
3. on établit une **référence** : la meilleure régression à une seule variable ;
4. on écrit la solution du cours, $\hat{\vec\beta} = (X^\top X)^{-1} X^\top \vec y$, en numpy ;
5. on vérifie avec `scikit-learn` ;
6. on compare les deux modèles sur le graphe prédictions contre réalité ;
7. on apprend à lire les coefficients — et on découvre qu'ils sont trompeurs ;
8. on termine sur le **surapprentissage**, qui motivera la régularisation du cours.

> **Une mise en garde, à lire avant de commencer.** Ces données décrivent une catastrophe
> sanitaire réelle. Les décès déclarés dépendent aussi de la qualité du système de
> santé et du dépistage : un pays qui compte mal ses morts apparaîtra faussement épargné.
> Tout ce qu'on trouvera ici est une **corrélation**, jamais une cause. On y revient en §10.

---
## 0. Mise en place

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression

CHEMIN = "data/covid_owid_countries.csv"   # adaptez si besoin
print("numpy", np.__version__, "| pandas", pd.__version__)

---
## 1. Les données

Le fichier contient aussi des lignes agrégées (« World », « Europe », « High income »…)
qu'il faut retirer : elles n'ont pas de continent renseigné. On ne garde ensuite que les
colonnes qui nous intéressent, et les pays pour lesquels **toutes** sont disponibles.

In [ ]:
VARIABLES = ["median_age",                  # age median de la population
             "aged_65_older",               # % de plus de 65 ans
             "gdp_per_capita",              # PIB par habitant
             "life_expectancy",             # esperance de vie
             "human_development_index",     # IDH
             "hospital_beds_per_thousand",  # lits d'hopital pour 1000 hab.
             "diabetes_prevalence",         # % de diabetiques
             "population_density"]          # habitants au km2

CIBLE = "total_deaths_per_million"


def charger(chemin=CHEMIN):
    """Charge le fichier OWID, retire les agregats et les lignes incompletes."""
    d = pd.read_csv(chemin)
    d = d[d.continent.notna()]                       # enleve World, Europe, etc.
    d = d[["location"] + VARIABLES + [CIBLE]].dropna()
    return d.reset_index(drop=True)


pays = charger()
print(f"{len(pays)} pays exploitables, {len(VARIABLES)} variables explicatives\n")
print(pays.head(4).to_string(index=False))

In [ ]:
X = pays[VARIABLES].to_numpy(float)
y = pays[CIBLE].to_numpy(float)
print("X :", X.shape, "  (une ligne = un pays, une colonne = une variable)")
print("y :", y.shape)

print("\nordres de grandeur tres differents d'une colonne a l'autre :")
for i, v in enumerate(VARIABLES):
    print(f"   {v:28s} de {X[:, i].min():10,.1f} a {X[:, i].max():12,.1f}")

> **Question 1.** Regardez la dernière colonne du tableau ci-dessus. Le PIB par habitant
> se compte en dizaines de milliers, la prévalence du diabète en pourcents. Gardez ça en
> tête : on verra en §7 que cela rend les coefficients **impossibles à comparer** tels quels.

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(13, 5.5))
for ax, v in zip(axes.ravel(), VARIABLES):
    ax.scatter(pays[v], pays[CIBLE], s=14, alpha=.6, color="#1f4e9c")
    ax.set_xlabel(v, fontsize=8); ax.set_ylabel("deces / million", fontsize=8)
    ax.tick_params(labelsize=7); ax.grid(alpha=.3)
plt.tight_layout()

---
## 2. Train / test

Comme dans le TP précédent : 30 % des pays sont mis de côté et ne serviront qu'à mesurer.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=0)
print(f"train : {X_train.shape[0]} pays | test : {X_test.shape[0]} pays")

---
## 3. La référence : une seule variable

Avant d'ajouter des variables, mesurons ce qu'on obtient avec **une seule**. Voici votre
fonction du TP précédent et le $R^2$, fournis ici pour gagner du temps.

In [ ]:
def estimer_droite(x, y):
    """Regression simple, formule du cours (votre fonction du TP precedent)."""
    xb, yb = x.mean(), y.mean()
    b1 = ((x - xb) * (y - yb)).sum() / ((x - xb) ** 2).sum()
    return yb - b1 * xb, b1


def r2(y_vrai, y_pred):
    """Coefficient de determination."""
    return 1 - ((y_vrai - y_pred) ** 2).sum() / ((y_vrai - y_vrai.mean()) ** 2).sum()


print(f"{'variable seule':30s} {'R2 test':>8s}")
scores_simples = {}
for i, v in enumerate(VARIABLES):
    b0, b1 = estimer_droite(X_train[:, i], y_train)
    scores_simples[v] = r2(y_test, b0 + b1 * X_test[:, i])
    print(f"   {v:28s} {scores_simples[v]:7.3f}")

MEILLEURE = max(scores_simples, key=scores_simples.get)
R2_simple = scores_simples[MEILLEURE]
print(f"\nmeilleure variable seule : {MEILLEURE} (R2 = {R2_simple:.3f})")

> **Question 2.** L'âge médian explique à lui seul une bonne partie de la mortalité,
> alors que le **PIB par habitant** et la **prévalence du diabète** n'expliquent
> quasiment rien ($R^2$ proche de zéro, voire négatif). Cela vous surprend-il ? Que
> pourrait-on en conclure — et que ne peut-on **pas** en conclure ?

---
## 4. La matrice de design

Avec $p$ variables, le modèle s'écrit

$$
y = \beta_0 + \beta_1 x_1 + \beta_2 x_2 + \dots + \beta_p x_p .
$$

Pour manipuler tout ça d'un coup, on range les observations dans une matrice $X$ où
**chaque ligne est un pays**. Petite astuce indispensable : on ajoute une **colonne de
1** tout à gauche, qui porte l'ordonnée à l'origine $\beta_0$ :

$$
X = \begin{pmatrix}
1 & x_{1,1} & \cdots & x_{1,p} \\
1 & x_{2,1} & \cdots & x_{2,p} \\
\vdots & \vdots & & \vdots \\
1 & x_{N,1} & \cdots & x_{N,p}
\end{pmatrix}
\qquad\text{de sorte que}\qquad
\hat{\vec y} = X \hat{\vec\beta}.
$$

**À vous.** Écrivez la fonction qui ajoute cette colonne de 1. Regardez du côté de
`np.ones` et `np.column_stack`.

In [ ]:
def matrice_design(X):
    """Ajoute une colonne de 1 a gauche de X.

    Args:
        X (np.ndarray): de forme (N, p)

    Returns:
        np.ndarray de forme (N, p+1)
    """
    # TODO
    raise NotImplementedError("a completer")

In [ ]:
# --- test ---
Mt = matrice_design(np.array([[2., 3.], [4., 5.]]))
assert Mt.shape == (2, 3), Mt.shape
assert np.allclose(Mt, [[1., 2., 3.], [1., 4., 5.]]), Mt
print("matrice_design : OK")
print(matrice_design(X_train)[:3])

---
## 5. Les équations normales

Le cours donne la solution du problème des moindres carrés :

$$
\hat{\vec\beta} = (X^\top X)^{-1} X^\top \vec y .
$$

**Un point important de mise en œuvre.** On n'inverse *jamais* une matrice en pratique :
c'est coûteux et numériquement instable. On résout plutôt le système équivalent

$$
(X^\top X)\, \hat{\vec\beta} = X^\top \vec y
$$

avec `np.linalg.solve(A, b)`, qui résout $A\beta = b$ sans calculer $A^{-1}$.

**À vous.** En numpy, le produit matriciel s'écrit `@` et la transposée `.T`.

In [ ]:
def estimer_multi(X, y):
    """Resout les equations normales.

    Returns:
        beta (np.ndarray) de taille p+1 : beta[0] est l'ordonnee a l'origine
    """
    M = matrice_design(X)
    # TODO : resoudre (M^T M) beta = M^T y avec np.linalg.solve
    raise NotImplementedError("a completer")


def predire_multi(beta, X):
    """Applique le modele : y_chapeau = X beta (avec la colonne de 1)."""
    # TODO
    raise NotImplementedError("a completer")

In [ ]:
# --- test sur des donnees ou la reponse est exacte ---
Xt = np.array([[1., 0.], [0., 1.], [1., 1.], [2., 3.]])
yt = 5. + 2. * Xt[:, 0] - 3. * Xt[:, 1]          # exactement y = 5 + 2x1 - 3x2
bt = estimer_multi(Xt, yt)
assert np.allclose(bt, [5., 2., -3.]), bt
assert np.allclose(predire_multi(bt, Xt), yt)

beta = estimer_multi(X_train, y_train)
y_pred_multi = predire_multi(beta, X_test)
R2_multi = r2(y_test, y_pred_multi)
print(f"R2 multiple : {R2_multi:.3f}   (contre {R2_simple:.3f} avec la seule variable {MEILLEURE})")

---
## 6. Vérification avec scikit-learn

`LinearRegression` gère la colonne de 1 toute seule (`fit_intercept=True` par défaut) :
on lui passe donc `X` **sans** la colonne ajoutée.

In [ ]:
modele = LinearRegression().fit(X_train, y_train)
beta_sk = np.concatenate([[modele.intercept_], modele.coef_])

print(f"{'':30s} {'a la main':>14s} {'sklearn':>14s}")
print(f"   {'ordonnee a l origine':28s} {beta[0]:14.4f} {beta_sk[0]:14.4f}")
for i, v in enumerate(VARIABLES):
    print(f"   {v:28s} {beta[i+1]:14.4f} {beta_sk[i+1]:14.4f}")
print(f"\necart maximal : {np.abs(beta - beta_sk).max():.2e}")
assert np.allclose(beta, beta_sk), "les deux methodes doivent coincider"
print("sklearn resout bien les memes equations normales : OK")

---
## 7. Prédictions contre réalité

Même graphe que dans le TP précédent, pour les deux modèles côte à côte.

In [ ]:
def graphe_diagonal(y_vrai, y_pred, titre="", ax=None, couleur="#c0392b"):
    """Nuage predictions vs realite, avec la diagonale de reference. Fourni."""
    if ax is None:
        _, ax = plt.subplots(figsize=(4.2, 4.2))
    lo = min(y_vrai.min(), y_pred.min()); hi = max(y_vrai.max(), y_pred.max())
    m = 0.05 * (hi - lo)
    ax.plot([lo-m, hi+m], [lo-m, hi+m], "--", color="gray", lw=1, label="modele parfait")
    ax.scatter(y_vrai, y_pred, s=40, color=couleur, zorder=3, edgecolor="white")
    ax.set_xlabel("deces/million observes"); ax.set_ylabel("deces/million predits")
    ax.set_title(titre, fontsize=10); ax.legend(fontsize=8, loc="upper left"); ax.grid(alpha=.3)
    return ax


i_best = VARIABLES.index(MEILLEURE)
b0, b1 = estimer_droite(X_train[:, i_best], y_train)
y_pred_simple = b0 + b1 * X_test[:, i_best]

fig, axes = plt.subplots(1, 2, figsize=(9, 4.4))
graphe_diagonal(y_test, y_pred_simple, f"1 variable ({MEILLEURE})\n$R^2$ = {R2_simple:.3f}",
                axes[0], "#c0392b")
graphe_diagonal(y_test, y_pred_multi, f"{len(VARIABLES)} variables\n$R^2$ = {R2_multi:.3f}",
                axes[1], "#1f4e9c")
plt.tight_layout()

> **Question 3.** Les points se resserrent-ils autour de la diagonale ? Repérez
> quelques pays très mal prédits par les deux modèles. La cellule suivante vous donne
> leurs noms.

In [ ]:
_, idx_test = train_test_split(np.arange(len(X)), test_size=0.3, random_state=0)
erreurs = pd.DataFrame({
    "pays": pays.location.to_numpy()[idx_test],
    "observe": y_test,
    "predit": y_pred_multi,
    "erreur": y_pred_multi - y_test,
}).sort_values("erreur")
print("--- les 5 pays les plus SOUS-estimes ---")
print(erreurs.head(5).to_string(index=False, float_format=lambda v: f"{v:8.0f}"))
print("\n--- les 5 pays les plus SUR-estimes ---")
print(erreurs.tail(5).to_string(index=False, float_format=lambda v: f"{v:8.0f}"))

---
## 8. Lire les coefficients — et le piège

On pourrait croire qu'un gros coefficient signale une variable importante. C'est faux
tant que les variables n'ont pas la **même échelle** : un coefficient s'exprime « par
unité de la variable », et une unité de PIB par habitant (1 dollar) n'a rien à voir avec
une unité d'âge médian (1 an).

La solution est de **standardiser** : on remplace chaque colonne par
$$
\tilde x = \frac{x - \bar x}{\sigma_x},
$$
de sorte que toutes les variables soient sans unité, de moyenne 0 et d'écart-type 1. Les
coefficients deviennent alors comparables entre eux.

**À vous.**

> Attention à un point de méthode : la moyenne et l'écart-type doivent être calculés
> **sur le train uniquement**, puis appliqués tels quels au test. Sinon le test aurait
> une influence sur l'entraînement — c'est une fuite de données.

In [ ]:
def standardiser(X_train, X_test):
    """Centre et reduit les colonnes, avec les statistiques du TRAIN seulement.

    Returns:
        (X_train_std, X_test_std)
    """
    # TODO : calculer moyenne et ecart-type sur X_train (axis=0),
    #        puis les appliquer aux deux tableaux
    raise NotImplementedError("a completer")

In [ ]:
# --- test ---
Xa = np.array([[0., 10.], [2., 20.], [4., 30.]])
Xb = np.array([[2., 20.]])
Sa, Sb = standardiser(Xa, Xb)
assert np.allclose(Sa.mean(axis=0), 0), Sa.mean(axis=0)
assert np.allclose(Sa.std(axis=0), 1), Sa.std(axis=0)
assert np.allclose(Sb, [[0., 0.]]), Sb          # la moyenne du train devient 0
print("standardiser : OK\n")

Xtr_std, Xte_std = standardiser(X_train, X_test)
beta_std = estimer_multi(Xtr_std, y_train)
print(f"R2 avec variables standardisees : {r2(y_test, predire_multi(beta_std, Xte_std)):.3f}"
      f"   (identique : standardiser ne change pas les predictions)\n")

ordre = np.argsort(-np.abs(beta_std[1:]))
print(f"{'variable':30s} {'coef brut':>14s} {'coef standardise':>18s}")
for i in ordre:
    print(f"   {VARIABLES[i]:28s} {beta[i+1]:14.4f} {beta_std[i+1]:18.1f}")

> **Question 4.** Comparez les deux colonnes pour `gdp_per_capita`. Son coefficient brut
> est minuscule — de l'ordre de $-0{,}02$ — ce qui donnerait l'impression d'une variable
> négligeable. Son coefficient standardisé est au contraire l'un des plus grands. D'où
> vient la contradiction ?
>
> **Question 5.** Le $R^2$ est strictement le même avec et sans standardisation.
> Pourquoi était-ce prévisible ? (Que fait-on vraiment aux données ?)
>
> **Question 6.** Plusieurs variables sont fortement liées entre elles : l'âge médian, la
> part des plus de 65 ans et l'espérance de vie mesurent un peu la même chose. Regardez
> leurs coefficients standardisés : certains sont positifs, d'autres négatifs, et tous
> sont grands. Quand des variables sont redondantes, le modèle peut leur attribuer des
> coefficients énormes qui se compensent — c'est la **colinéarité**, et c'est l'une des
> raisons d'être de la régression Ridge que vous verrez en cours.

---
## 9. Ajouter des variables améliore-t-il toujours ?

Ajoutons au modèle des colonnes de **bruit pur** — des nombres tirés au hasard, sans
aucun rapport avec la mortalité — et regardons ce qui arrive au $R^2$ mesuré sur le
train et sur le test.

In [ ]:
rng = np.random.default_rng(0)
print(f"{'colonnes de bruit ajoutees':>28s} {'R2 train':>10s} {'R2 test':>11s}")
for k in (0, 5, 20, 50, 100):
    Xa = np.column_stack([X_train, rng.normal(size=(len(X_train), k))]) if k else X_train
    Xb = np.column_stack([X_test, rng.normal(size=(len(X_test), k))]) if k else X_test
    bk = estimer_multi(Xa, y_train)
    print(f"{k:28d} {r2(y_train, predire_multi(bk, Xa)):10.3f} "
          f"{r2(y_test, predire_multi(bk, Xb)):11.3f}")

> **Question 7.** Le $R^2$ **sur le train** ne cesse de monter : avec 100 colonnes de
> bruit il frôle 1, le modèle « explique » presque parfaitement les données
> d'entraînement. Pendant ce temps, le $R^2$ **sur le test** s'effondre jusqu'à devenir
> très négatif. Expliquez ce qui se passe.
>
> (Note : au départ, le $R^2$ de test est ici un peu supérieur à celui du train. Ce n'est
> pas anormal avec seulement 112 pays d'entraînement — c'est le hasard du découpage. Ce
> qui compte est l'**écart** entre les deux courbes, qui ne cesse de se creuser.)
>
> **Question 8.** Avec 112 pays d'entraînement et 108 variables, combien le modèle
> a-t-il de paramètres à ajuster par rapport au nombre d'observations ? Que se
> passerait-il avec 112 variables exactement ?
>
> **Question 9.** On ne peut donc pas se contenter d'ajouter des variables. Deux familles
> de solutions existent : en **sélectionner** un sous-ensemble, ou **pénaliser** les gros
> coefficients pour empêcher le modèle de s'emballer. C'est exactement ce que font les
> régressions **Ridge** et **LASSO** du cours.

---
## 10. Ce que ces données ne disent pas

Le modèle trouve une association forte entre la structure par âge d'un pays et sa
mortalité déclarée. Avant d'en tirer la moindre conclusion, trois réserves.

**Les décès déclarés ne sont pas les décès réels.** Un pays qui teste peu et dont l'état
civil est incomplet déclarera moins de décès — et apparaîtra donc, à tort, comme épargné.
Les pays les plus « sur-estimés » par le modèle en §7 sont souvent dans ce cas.

**Corrélation n'est pas causalité.** Que l'âge médian soit associé à la mortalité est
cohérent avec ce qu'on sait du virus, mais le modèle, lui, ne démontre rien : il mesure
une association sur 160 points, sans notion de mécanisme.

**Les pays ne sont pas des observations indépendantes.** Voisins, ils partagent
climat, politiques et calendrier épidémique. L'hypothèse d'indépendance des résidus,
sous-jacente à la régression, est douteuse ici.

> **Question 10.** Parmi les pays les plus mal prédits que vous avez listés en §7,
> cherchez-en deux ou trois. Leurs particularités relèvent-elles du modèle, ou de la
> façon dont les données ont été collectées ?

---
## 11. Pour aller plus loin (facultatif)

- Changez `CIBLE` pour `total_cases_per_million` : les mêmes variables expliquent-elles
  aussi bien les cas que les décès ?
- Retirez `median_age` de `VARIABLES` et ré-entraînez. Le $R^2$ s'effondre-t-il, ou les
  autres variables compensent-elles ? Que dit ce résultat sur la colinéarité ?
- Le fichier contient bien d'autres colonnes (`pd.read_csv(CHEMIN).columns`) : essayez
  d'en ajouter.

In [ ]:
# TODO (facultatif) : votre exploration ici

---
### Ce qu'il faut retenir

- Le passage d'une à plusieurs variables ne change pas la **méthode** : on annule encore
  les dérivées de la RSS, ce qui donne cette fois un système linéaire,
  $\hat{\vec\beta} = (X^\top X)^{-1} X^\top \vec y$.
- En pratique on ne l'inverse pas : on **résout** le système avec `np.linalg.solve`.
- Huit variables font mieux qu'une seule ici — mais la meilleure variable seule n'était
  pas loin. Ajouter des variables ne garantit rien.
- Les coefficients ne sont comparables qu'une fois les variables **standardisées**, et
  la standardisation se calcule sur le **train** uniquement.
- Des variables redondantes donnent des coefficients énormes qui se compensent : c'est la
  **colinéarité**.
- Trop de variables pour trop peu d'observations mène au **surapprentissage** : $R^2$
  parfait en train, catastrophique en test. C'est le problème que **Ridge** et **LASSO**
  viennent résoudre.